In [ ]:
import numpy as np
from iminuit import Minuit
import flavio
from flavio.statistics.likelihood import FastLikelihood
from flavio.classes import Measurement
from flavio.statistics.probability import MultivariateNormalDistribution
import random
import os

# --- CONGELACIÓN DE SEMILLAS ---
i_seed = 42
os.environ['PYTHONHASHSEED'] = str(i_seed)
random.seed(i_seed)
np.random.seed(i_seed)

observables = [
    'BR(Bs->mumu)', 'BR(B0->mumu)',
    ('<Rmue>(B+->Kll)', 1.1, 6.0), ('<Rmue>(B0->K*ll)', 0.045, 1.1), ('<Rmue>(B0->K*ll)', 1.1, 6.0),
    ('<P5p>(B0->K*mumu)', 4.0, 6.0),
    ('<dBR/dq2>(B+->Kmumu)', 1.1, 6.0), ('<dBR/dq2>(Bs->phimumu)', 1.1, 6.0)
]

# 1. INPUTS EXPERIMENTALES
datos_reales = {
    'BR(Bs->mumu)': {'c': 3.83e-9, 'e': np.sqrt(0.38**2 + 0.19**2 + 0.14**2) * 1e-9},
    'BR(B0->mumu)': {'c': 1.20e-10, 'e': np.sqrt(0.83**2 + 0.14**2) * 1e-10},
    ('<Rmue>(B+->Kll)', 1.1, 6.0): {'c': 0.949, 'e': np.sqrt(0.042**2 + 0.022**2)},
    ('<Rmue>(B0->K*ll)', 0.045, 1.1): {'c': 0.927, 'e': np.sqrt(0.093**2 + 0.036**2)},
    ('<Rmue>(B0->K*ll)', 1.1, 6.0): {'c': 1.027, 'e': np.sqrt(0.072**2 + 0.027**2)},
    ('<P5p>(B0->K*mumu)', 4.0, 6.0): {'c': -0.439, 'e': np.sqrt(0.111**2 + 0.036**2)},
    ('<dBR/dq2>(B+->Kmumu)', 1.1, 6.0): {'c': 1.19e-8, 'e': np.sqrt(0.03**2 + 0.06**2) * 1e-8},
    ('<dBR/dq2>(Bs->phimumu)', 1.1, 6.0): {'c': (2.88 / 4.9) * 1e-8, 'e': (np.sqrt(0.15**2 + 0.05**2 + 0.14**2) / 4.9) * 1e-8}
}

mean_vec = np.array([datos_reales[obs]['c'] for obs in observables])
err_vec = np.array([datos_reales[obs]['e'] for obs in observables])

# 2. MATRIZ DE COVARIANZA EXPERIMENTAL ARTESANAL
cov_matrix_exp = np.diag(err_vec**2)

# A) Correlación B_s -> mumu y B^0 -> mumu
cov_matrix_exp[0, 1] = cov_matrix_exp[1, 0] = -0.11 * err_vec[0] * err_vec[1] 

# B) Correlaciones sistemáticas en LFU (R_K y R_K*)
cov_matrix_exp[2, 4] = cov_matrix_exp[4, 2] = 0.03 * err_vec[2] * err_vec[4] 
cov_matrix_exp[3, 4] = cov_matrix_exp[4, 3] = 0.05 * err_vec[3] * err_vec[4] 

# 3. AISLAR FLAVIO (Borramos las medidas mundiales de la memoria RAM)
flavio.Measurement.instances.clear()

# 4. INYECTAR LOS INPUTS MULTIVARIADOS
m_tfg = Measurement('Datos_TFG_Custom')
obs_names = [obs for obs in observables]
m_tfg.add_constraint(obs_names, MultivariateNormalDistribution(mean_vec, cov_matrix_exp))

# 5. EJECUTAR FASTLIKELIHOOD SÓLO CON LOS DATOS AISLADOS
fl = FastLikelihood('Ajuste_Aislado', observables=observables, nuisance_parameters='all')
fl.make_measurement(N=5000, Nexp=5000, threads=1)

par_central = flavio.default_parameters.get_central_all()

def loglike_C9(C9, scale=5.0):
    wc = flavio.WilsonCoefficients()
    wc.set_initial({'C9_bsmumu': C9}, scale)
    return fl.log_likelihood(par_central, wc)

def nll(C9):
    return -loglike_C9(C9)

# 6. MINIMIZACIÓN CLÁSICA
m = Minuit(nll, C9=0.0)
m.errordef = 0.5 
m.migrad()
m.hesse()
m.minos()

print("\n=== Best-fit parameter Minuit (Con matriz customizada) ===")
print(f"C9 = {m.values['C9']:.6f} +{m.merrors['C9'].upper:.6f} {m.merrors['C9'].lower:.6f}")


=== Best-fit parameter Minuit (Con matriz customizada) ===
C9 = -1.252377 +0.190324 -0.199280


In [ ]:
import numpy as np
from iminuit import Minuit
import flavio
from flavio.statistics.likelihood import FastLikelihood
from flavio.classes import Measurement
from flavio.statistics.probability import MultivariateNormalDistribution
import random
import os

# --- CONGELACIÓN DE SEMILLAS ---
i_seed = 42
os.environ['PYTHONHASHSEED'] = str(i_seed)
random.seed(i_seed)
np.random.seed(i_seed)

observables = [
    'BR(Bs->mumu)', 'BR(B0->mumu)',
    ('<Rmue>(B+->Kll)', 1.1, 6.0), ('<Rmue>(B0->K*ll)', 0.045, 1.1), ('<Rmue>(B0->K*ll)', 1.1, 6.0),
    ('<P5p>(B0->K*mumu)', 4.0, 6.0),
    ('<dBR/dq2>(B+->Kmumu)', 1.1, 6.0), ('<dBR/dq2>(Bs->phimumu)', 1.1, 6.0)
]

# 1. INPUTS EXPERIMENTALES
datos_reales = {
    'BR(Bs->mumu)': {'c': 3.83e-9, 'e': np.sqrt(0.38**2 + 0.19**2 + 0.14**2) * 1e-9},
    'BR(B0->mumu)': {'c': 1.20e-10, 'e': np.sqrt(0.83**2 + 0.14**2) * 1e-10},
    ('<Rmue>(B+->Kll)', 1.1, 6.0): {'c': 0.949, 'e': np.sqrt(0.042**2 + 0.022**2)},
    ('<Rmue>(B0->K*ll)', 0.045, 1.1): {'c': 0.927, 'e': np.sqrt(0.093**2 + 0.036**2)},
    ('<Rmue>(B0->K*ll)', 1.1, 6.0): {'c': 1.027, 'e': np.sqrt(0.072**2 + 0.027**2)},
    ('<P5p>(B0->K*mumu)', 4.0, 6.0): {'c': -0.439, 'e': np.sqrt(0.111**2 + 0.036**2)},
    ('<dBR/dq2>(B+->Kmumu)', 1.1, 6.0): {'c': 1.19e-8, 'e': np.sqrt(0.03**2 + 0.06**2) * 1e-8},
    ('<dBR/dq2>(Bs->phimumu)', 1.1, 6.0): {'c': (2.88 / 4.9) * 1e-8, 'e': (np.sqrt(0.15**2 + 0.05**2 + 0.14**2) / 4.9) * 1e-8}
}

mean_vec = np.array([datos_reales[obs]['c'] for obs in observables])
err_vec = np.array([datos_reales[obs]['e'] for obs in observables])

# 2. MATRIZ DE COVARIANZA EXPERIMENTAL
cov_matrix_exp = np.diag(err_vec**2)
cov_matrix_exp[0, 1] = cov_matrix_exp[1, 0] = -0.11 * err_vec[0] * err_vec[1] 
cov_matrix_exp[2, 4] = cov_matrix_exp[4, 2] = 0.03 * err_vec[2] * err_vec[4] 
cov_matrix_exp[3, 4] = cov_matrix_exp[4, 3] = 0.05 * err_vec[3] * err_vec[4] 

# 3. AISLAR FLAVIO
flavio.Measurement.instances.clear()

# 4. INYECTAR INPUTS
m_tfg = Measurement('Datos_TFG_Custom')
m_tfg.add_constraint(observables, MultivariateNormalDistribution(mean_vec, cov_matrix_exp))

# 5. FASTLIKELIHOOD
fl = FastLikelihood('Ajuste_Aislado', observables=observables, nuisance_parameters='all')
fl.make_measurement(N=2000, Nexp=5000, threads=1)

par_central = flavio.default_parameters.get_central_all()

# --- CAMBIOS PARA C9 Y C10 ---

def loglike_Wilson(C9, C10, scale=4.8):
    wc = flavio.WilsonCoefficients()
    # Definimos las contribuciones de Nueva Física (NP)
    wc.set_initial({'C9_bsmumu': C9, 'C10_bsmumu': C10}, scale)
    return fl.log_likelihood(par_central, wc)

def nll(C9, C10):
    return -loglike_Wilson(C9, C10)

# 6. MINIMIZACIÓN BIDIMENSIONAL
# Iniciamos en 0.0 (SM) para ambos
m = Minuit(nll, C9=0.0, C10=0.0)
m.errordef = 0.5 
m.migrad()
m.hesse()
m.minos()

print("\n=== Best-fit parameters Minuit (Ajuste C9 y C10) ===")
for p in ['C9', 'C10']:
    val = m.values[p]
    up = m.merrors[p].upper
    lo = m.merrors[p].lower
    print(f"{p} = {val:.4f} +{up:.4f} {lo:.4f}")

print("\nMatriz de correlación:")
print(m.covariance.correlation())


=== Best-fit parameters Minuit (Ajuste C9 y C10) ===
C9 = -1.2778 +0.2740 -0.2777
C10 = -0.0167 +0.1680 -0.1563

Matriz de correlación:
┌─────┬─────────┐
│     │  C9 C10 │
├─────┼─────────┤
│  C9 │   1 0.7 │
│ C10 │ 0.7   1 │
└─────┴─────────┘


In [ ]:
import numpy as np
from iminuit import Minuit
import flavio
from flavio.statistics.likelihood import FastLikelihood
from flavio.classes import Measurement
from flavio.statistics.probability import MultivariateNormalDistribution
import random
import os

# --- CONGELACIÓN DE SEMILLAS ---
i_seed = 42
os.environ['PYTHONHASHSEED'] = str(i_seed)
random.seed(i_seed)
np.random.seed(i_seed)

observables = [
    'BR(Bs->mumu)', 'BR(B0->mumu)',
    ('<Rmue>(B+->Kll)', 1.1, 6.0), ('<Rmue>(B0->K*ll)', 0.045, 1.1), ('<Rmue>(B0->K*ll)', 1.1, 6.0),
    ('<P5p>(B0->K*mumu)', 4.0, 6.0),
    ('<dBR/dq2>(B+->Kmumu)', 1.1, 6.0), ('<dBR/dq2>(Bs->phimumu)', 1.1, 6.0)
]

# 1. INPUTS EXPERIMENTALES
datos_reales = {
    'BR(Bs->mumu)': {'c': 3.83e-9, 'e': np.sqrt(0.38**2 + 0.19**2 + 0.14**2) * 1e-9},
    'BR(B0->mumu)': {'c': 1.20e-10, 'e': np.sqrt(0.83**2 + 0.14**2) * 1e-10},
    ('<Rmue>(B+->Kll)', 1.1, 6.0): {'c': 0.949, 'e': np.sqrt(0.042**2 + 0.022**2)},
    ('<Rmue>(B0->K*ll)', 0.045, 1.1): {'c': 0.927, 'e': np.sqrt(0.093**2 + 0.036**2)},
    ('<Rmue>(B0->K*ll)', 1.1, 6.0): {'c': 1.027, 'e': np.sqrt(0.072**2 + 0.027**2)},
    ('<P5p>(B0->K*mumu)', 4.0, 6.0): {'c': -0.439, 'e': np.sqrt(0.111**2 + 0.036**2)},
    ('<dBR/dq2>(B+->Kmumu)', 1.1, 6.0): {'c': 1.19e-8, 'e': np.sqrt(0.03**2 + 0.06**2) * 1e-8},
    ('<dBR/dq2>(Bs->phimumu)', 1.1, 6.0): {'c': (2.88 / 4.9) * 1e-8, 'e': (np.sqrt(0.15**2 + 0.05**2 + 0.14**2) / 4.9) * 1e-8}
}

mean_vec = np.array([datos_reales[obs]['c'] for obs in observables])
err_vec = np.array([datos_reales[obs]['e'] for obs in observables])

# 2. MATRIZ DE COVARIANZA EXPERIMENTAL
cov_matrix_exp = np.diag(err_vec**2)
cov_matrix_exp[0, 1] = cov_matrix_exp[1, 0] = -0.11 * err_vec[0] * err_vec[1] 
cov_matrix_exp[2, 4] = cov_matrix_exp[4, 2] = 0.03 * err_vec[2] * err_vec[4] 
cov_matrix_exp[3, 4] = cov_matrix_exp[4, 3] = 0.05 * err_vec[3] * err_vec[4] 

# 3. AISLAR FLAVIO
flavio.Measurement.instances.clear()

# 4. INYECTAR INPUTS
m_tfg = Measurement('Datos_TFG_Custom')
m_tfg.add_constraint(observables, MultivariateNormalDistribution(mean_vec, cov_matrix_exp))

# 5. FASTLIKELIHOOD
fl = FastLikelihood('Ajuste_Aislado', observables=observables, nuisance_parameters='all')
fl.make_measurement(N=2000, Nexp=5000, threads=1)

par_central = flavio.default_parameters.get_central_all()

# --- CAMBIOS PARA C9, C10, C9' Y C10' ---

def loglike_Wilson(C9, C10, C9p, C10p, scale=4.8):
    wc = flavio.WilsonCoefficients()
    # Definimos las contribuciones de Nueva Física (NP) para los 4 coeficientes
    wc.set_initial({
        'C9_bsmumu': C9, 
        'C10_bsmumu': C10, 
        'C9p_bsmumu': C9p, 
        'C10p_bsmumu': C10p
    }, scale)
    return fl.log_likelihood(par_central, wc)

def nll(C9, C10, C9p, C10p):
    return -loglike_Wilson(C9, C10, C9p, C10p)

# 6. MINIMIZACIÓN CUATRIDIMENSIONAL
# Iniciamos en 0.0 (Standard Model) para los 4 coeficientes
m = Minuit(nll, C9=0.0, C10=0.0, C9p=0.0, C10p=0.0)
m.errordef = 0.5 
m.migrad()
m.hesse()
m.minos()

print("\n=== Best-fit parameters Minuit (Ajuste 4D: C9, C10, C9', C10') ===")
for p in ['C9', 'C10', 'C9p', 'C10p']:
    val = m.values[p]
    # m.merrors puede fallar si minos() no converge perfectamente en 4D
    # Usamos un try-except sencillo o .valid por si acaso
    try:
        up = m.merrors[p].upper
        lo = m.merrors[p].lower
        print(f"{p} = {val:.4f} +{up:.4f} {lo:.4f}")
    except KeyError:
        # Si MINOS falla, imprimimos el error parabólico de HESSE
        err = m.errors[p]
        print(f"{p} = {val:.4f} ± {err:.4f} (Hesse error)")

print("\nMatriz de correlación:")
print(m.covariance.correlation())


=== Best-fit parameters Minuit (Ajuste 4D: C9, C10, C9', C10') ===
C9 = -1.8997 +0.6052 -0.5680
C10 = 2.5317 +0.9766 -1.3177
C9p = 2.8526 +0.3791 -0.5379
C10p = 2.7441 +0.8794 -1.2184

Matriz de correlación:
┌──────┬─────────────────────┐
│      │   C9  C10  C9p C10p │
├──────┼─────────────────────┤
│   C9 │    1 -0.7 -0.2 -0.6 │
│  C10 │ -0.7    1 -0.6    1 │
│  C9p │ -0.2 -0.6    1 -0.6 │
│ C10p │ -0.6    1 -0.6    1 │
└──────┴─────────────────────┘


In [ ]:
import numpy as np
from iminuit import Minuit
import flavio
from flavio.statistics.likelihood import FastLikelihood
from flavio.classes import Measurement
from flavio.statistics.probability import MultivariateNormalDistribution
import random
import os

# --- CONGELACIÓN DE SEMILLAS ---
i_seed = 42
os.environ['PYTHONHASHSEED'] = str(i_seed)
random.seed(i_seed)
np.random.seed(i_seed)

observables = [
    'BR(Bs->mumu)', 'BR(B0->mumu)',
    ('<Rmue>(B+->Kll)', 1.1, 6.0), ('<Rmue>(B0->K*ll)', 0.045, 1.1), ('<Rmue>(B0->K*ll)', 1.1, 6.0),
    ('<P5p>(B0->K*mumu)', 4.0, 6.0),
    ('<dBR/dq2>(B+->Kmumu)', 1.1, 6.0), ('<dBR/dq2>(Bs->phimumu)', 1.1, 6.0)
]

# 1. INPUTS EXPERIMENTALES
datos_reales = {
    'BR(Bs->mumu)': {'c': 3.83e-9, 'e': np.sqrt(0.38**2 + 0.19**2 + 0.14**2) * 1e-9},
    'BR(B0->mumu)': {'c': 1.20e-10, 'e': np.sqrt(0.83**2 + 0.14**2) * 1e-10},
    ('<Rmue>(B+->Kll)', 1.1, 6.0): {'c': 0.949, 'e': np.sqrt(0.042**2 + 0.022**2)},
    ('<Rmue>(B0->K*ll)', 0.045, 1.1): {'c': 0.927, 'e': np.sqrt(0.093**2 + 0.036**2)},
    ('<Rmue>(B0->K*ll)', 1.1, 6.0): {'c': 1.027, 'e': np.sqrt(0.072**2 + 0.027**2)},
    ('<P5p>(B0->K*mumu)', 4.0, 6.0): {'c': -0.439, 'e': np.sqrt(0.111**2 + 0.036**2)},
    ('<dBR/dq2>(B+->Kmumu)', 1.1, 6.0): {'c': 1.19e-8, 'e': np.sqrt(0.03**2 + 0.06**2) * 1e-8},
    ('<dBR/dq2>(Bs->phimumu)', 1.1, 6.0): {'c': (2.88 / 4.9) * 1e-8, 'e': (np.sqrt(0.15**2 + 0.05**2 + 0.14**2) / 4.9) * 1e-8}
}

mean_vec = np.array([datos_reales[obs]['c'] for obs in observables])
err_vec = np.array([datos_reales[obs]['e'] for obs in observables])

# 2. MATRIZ DE COVARIANZA
cov_matrix_exp = np.diag(err_vec**2)
cov_matrix_exp[0, 1] = cov_matrix_exp[1, 0] = -0.11 * err_vec[0] * err_vec[1] 
cov_matrix_exp[2, 4] = cov_matrix_exp[4, 2] = 0.03 * err_vec[2] * err_vec[4] 
cov_matrix_exp[3, 4] = cov_matrix_exp[4, 3] = 0.05 * err_vec[3] * err_vec[4] 

# 3. AISLAR FLAVIO
flavio.Measurement.instances.clear()

# 4. INYECTAR INPUTS
m_tfg = Measurement('Datos_TFG_6D')
m_tfg.add_constraint(observables, MultivariateNormalDistribution(mean_vec, cov_matrix_exp))

# 5. FASTLIKELIHOOD
fl = FastLikelihood('Ajuste_6D', observables=observables, nuisance_parameters='all')
# Aumentar N/Nexp mejora la precisión en espacios de alta dimensionalidad (6D)
fl.make_measurement(N=2000, Nexp=5000, threads=1)

par_central = flavio.default_parameters.get_central_all()

# --- AJUSTE HEXADIMENSIONAL ---

def loglike_Wilson(C9, C10, C9p, C10p, CS, CP, scale=4.8):
    wc = flavio.WilsonCoefficients()
    wc.set_initial({
        'C9_bsmumu': C9, 
        'C10_bsmumu': C10, 
        'C9p_bsmumu': C9p, 
        'C10p_bsmumu': C10p,
        'CS_bsmumu': CS,   
        'CP_bsmumu': CP
    }, scale)
    return fl.log_likelihood(par_central, wc)

def nll(C9, C10, C9p, C10p, CS, CP):
    return -loglike_Wilson(C9, C10, C9p, C10p, CS, CP)

# 6. MINIMIZACIÓN 6D
# Iniciamos todos en 0.0 (Standard Model)
params = ['C9', 'C10', 'C9p', 'C10p', 'CS', 'CP']
m = Minuit(nll, C9=0.0, C10=0.0, C9p=0.0, C10p=0.0, CS=0.0, CP=0.0)
m.errordef = 0.5 

print("Ejecutando MIGRAD (6D)...")
m.migrad()
m.hesse()

# MINOS en 6D puede ser inestable si los observables no restringen bien ciertos parámetros
print("Ejecutando MINOS...")
m.minos()

print("\n=== Best-fit parameters Minuit (Ajuste 6D) ===")
for p in params:
    val = m.values[p]
    try:
        up = m.merrors[p].upper
        lo = m.merrors[p].lower
        print(f"{p:5} = {val:8.4f} +{up:8.4f} {lo:8.4f}")
    except KeyError:
        err = m.errors[p]
        print(f"{p:5} = {val:8.4f} ± {err:8.4f} (Hesse error)")

print("\nMatriz de correlación (6x6):")
print(m.covariance.correlation())

Ejecutando MIGRAD (6D)...
Ejecutando MINOS...

=== Best-fit parameters Minuit (Ajuste 6D) ===
C9    =  -3.7991 +  0.8431  -0.4016
C10   =   5.9234 +  2.4722  -2.3497
C9p   =   3.7973 +  0.4140  -2.5911
C10p  =   1.3153 +  0.9394  -0.8672
CS    =  -0.0258 +  0.0367  -0.0115
CP    =  -0.0260 +  0.0841  -0.0351

Matriz de correlación (6x6):
┌──────┬───────────────────────────────┐
│      │   C9  C10  C9p C10p   CS   CP │
├──────┼───────────────────────────────┤
│   C9 │    1 -0.7  0.8  0.2  0.1  0.1 │
│  C10 │ -0.7    1 -0.9 -0.8 -0.3 -0.2 │
│  C9p │  0.8 -0.9    1  0.6  0.2  0.2 │
│ C10p │  0.2 -0.8  0.6    1  0.3  0.2 │
│   CS │  0.1 -0.3  0.2  0.3    1 -0.9 │
│   CP │  0.1 -0.2  0.2  0.2 -0.9    1 │
└──────┴───────────────────────────────┘
